In [1]:
# Chapter 4
## A Model for an Animal Evaluation (Animal Model)
# Accuracy of evaluations

In [56]:
using DataFrames
using LinearAlgebra

In [2]:
# Prepare pedigree
a = collect(1:8)
s = [missing, missing, missing, 1, 3, 1, 4, 3]
d = [missing, missing,missing,missing, 2, 2, 5, 6]

8-element Vector{Int64}:
 1
 2
 3
 4
 5
 6
 7
 8

In [10]:
ped = DataFrame(a=a,s=s,d=d)

Row,a,s,d
,Int64,Int64?,Int64?
1,1,missing,missing
2,2,missing,missing
3,3,missing,missing
4,4,1,missing
5,5,3,2
6,6,1,2
7,7,4,5
8,8,3,6


In [19]:
ped = coalesce.(ped, 0)

Row,a,s,d
,Int64,Int64,Int64
1,1,0,0
2,2,0,0
3,3,0,0
4,4,1,0
5,5,3,2
6,6,1,2
7,7,4,5
8,8,3,6


In [200]:
using NextGP
Ainv = inv(NextGP.makeA(ped.s,ped.d))

8×8 Matrix{Float64}:
  1.83333    0.5   0.0  -0.666667   0.0  -1.0   0.0  -0.0
  0.5        2.0   0.5  -0.0       -1.0  -1.0   0.0  -0.0
  0.0        0.5   2.0  -0.0       -1.0   0.5   0.0  -1.0
 -0.666667   0.0   0.0   1.83333    0.5   0.0  -1.0  -0.0
  0.0       -1.0  -1.0   0.5        2.5   0.0  -1.0  -0.0
 -1.0       -1.0   0.5   0.0        0.0   2.5   0.0  -1.0
  0.0        0.0   0.0  -1.0       -1.0   0.0   2.0  -0.0
  0.0        0.0  -1.0   0.0        0.0  -1.0   0.0   2.0

In [25]:
# Prepare data
wwg = [missing, missing, missing, 4.5, 2.9, 3.9, 3.5, 5.0]
sex = ["Male", "Female", "Male", "Male", "Female", "Female", "Male", "Male"]

data = DataFrame(wwg=wwg,sex=sex)

Row,wwg,sex
,Float64?,String
1,missing,Male
2,missing,Female
3,missing,Male
4,4.5,Male
5,2.9,Female
6,3.9,Female
7,3.5,Male
8,5.0,Male


In [26]:
data = hcat(ped,data)

Row,a,s,d,wwg,sex
,Int64,Int64,Int64,Float64?,String
1,1,0,0,missing,Male
2,2,0,0,missing,Female
3,3,0,0,missing,Male
4,4,1,0,4.5,Male
5,5,3,2,2.9,Female
6,6,1,2,3.9,Female
7,7,4,5,3.5,Male
8,8,3,6,5.0,Male


In [28]:
# Variance components 
varA = 20
varE = 40

40

In [99]:
# Variance ratios
α = varE/varA

2.0

In [30]:
# Setting up the incidence matrices for the MME

In [101]:
#rows with data
dataRows = .!ismissing.(data.wwg)

8-element BitVector:
 0
 0
 0
 1
 1
 1
 1
 1

In [145]:
means = ones(sum(dataRows))

5-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0

In [164]:
sex = [1 0;
       0 1;
       1 0;
       1 0;
       0 1;
       0 1;
       1 0;
       1 0]

8×2 Matrix{Int64}:
 1  0
 0  1
 1  0
 1  0
 0  1
 0  1
 1  0
 1  0

In [165]:
sex = sex[dataRows,:]

5×2 Matrix{Int64}:
 1  0
 0  1
 0  1
 1  0
 1  0

In [166]:
#X = [means sex]
X = sex

5×2 Matrix{Int64}:
 1  0
 0  1
 0  1
 1  0
 1  0

In [167]:
Z = diagm(dataRows)

8×8 BitMatrix:
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0
 0  0  0  1  0  0  0  0
 0  0  0  0  1  0  0  0
 0  0  0  0  0  1  0  0
 0  0  0  0  0  0  1  0
 0  0  0  0  0  0  0  1

In [168]:
Z = Z[dataRows,:]

5×8 BitMatrix:
 0  0  0  1  0  0  0  0
 0  0  0  0  1  0  0  0
 0  0  0  0  0  1  0  0
 0  0  0  0  0  0  1  0
 0  0  0  0  0  0  0  1

In [169]:
#y = data.wwg

8-element Vector{Union{Missing, Float64}}:
  missing
  missing
  missing
 4.5
 2.9
 3.9
 3.5
 5.0

In [196]:
#y = y.*dataRows
y = data.wwg[dataRows]

5-element Vector{Union{Missing, Float64}}:
 4.5
 2.9
 3.9
 3.5
 5.0

In [171]:
# Components of MME

In [172]:
X'X

2×2 Matrix{Int64}:
 3  0
 0  2

In [173]:
X'Z

2×8 Matrix{Int64}:
 0  0  0  1  0  0  1  1
 0  0  0  0  1  1  0  0

In [174]:
Z'X

8×2 Matrix{Int64}:
 0  0
 0  0
 0  0
 1  0
 0  1
 0  1
 1  0
 1  0

In [175]:
Z'Z+Ainv*α

8×8 Matrix{Float64}:
  3.66667   1.0   0.0  -1.33333   0.0  -2.0   0.0   0.0
  1.0       4.0   1.0   0.0      -2.0  -2.0   0.0   0.0
  0.0       1.0   4.0   0.0      -2.0   1.0   0.0  -2.0
 -1.33333   0.0   0.0   4.66667   1.0   0.0  -2.0   0.0
  0.0      -2.0  -2.0   1.0       6.0   0.0  -2.0   0.0
 -2.0      -2.0   1.0   0.0       0.0   6.0   0.0  -2.0
  0.0       0.0   0.0  -2.0      -2.0   0.0   5.0   0.0
  0.0       0.0  -2.0   0.0       0.0  -2.0   0.0   5.0

In [197]:
# MME

In [176]:
LHS = [X'X X'Z
       Z'X Z'Z+Ainv*α]

10×10 Matrix{Float64}:
 3.0  0.0   0.0       0.0   0.0   1.0       0.0   0.0   1.0   1.0
 0.0  2.0   0.0       0.0   0.0   0.0       1.0   1.0   0.0   0.0
 0.0  0.0   3.66667   1.0   0.0  -1.33333   0.0  -2.0   0.0   0.0
 0.0  0.0   1.0       4.0   1.0   0.0      -2.0  -2.0   0.0   0.0
 0.0  0.0   0.0       1.0   4.0   0.0      -2.0   1.0   0.0  -2.0
 1.0  0.0  -1.33333   0.0   0.0   4.66667   1.0   0.0  -2.0   0.0
 0.0  1.0   0.0      -2.0  -2.0   1.0       6.0   0.0  -2.0   0.0
 0.0  1.0  -2.0      -2.0   1.0   0.0       0.0   6.0   0.0  -2.0
 1.0  0.0   0.0       0.0   0.0  -2.0      -2.0   0.0   5.0   0.0
 1.0  0.0   0.0       0.0  -2.0   0.0       0.0  -2.0   0.0   5.0

In [177]:
RHS = [X'y;
       Z'y]

10-element Vector{Union{Missing, Float64}}:
 13.0
  6.8
  0.0
  0.0
  0.0
  4.5
  2.9
  3.9
  3.5
  5.0

In [178]:
sol = LHS\RHS

10-element Vector{Float64}:
  4.358502329854958
  3.4044300059066743
  0.09844457570387888
 -0.018770099100872753
 -0.04108420292708526
 -0.008663122661941079
 -0.185732099494651
  0.1768720876813023
 -0.24945855483362836
  0.18261468793069524

In [179]:
# Accuracy of evaluations

In [181]:
C = inv(LHS)

10×10 Matrix{Float64}:
  0.595561    0.157302  -0.164119    …  -0.166326   -0.284246  -0.237879
  0.157302    0.802459  -0.132863       -0.306003   -0.185949  -0.198649
 -0.164119   -0.132863   0.471094        0.220774    0.138622   0.134192
 -0.0836246  -0.241251   0.00692804      0.245156    0.119819   0.110664
 -0.130591   -0.111968   0.0326467       0.0226135   0.125898   0.217747
 -0.264557   -0.087308   0.219544    …   0.127572    0.242801   0.123191
 -0.148278   -0.298915   0.0449523       0.169723    0.219716   0.178074
 -0.166326   -0.306003   0.220774        0.442283    0.152183   0.219224
 -0.284246   -0.185949   0.138622        0.152183    0.441856   0.168082
 -0.237879   -0.198649   0.134192        0.219224    0.168082   0.422364

In [185]:
diagC = diag(C[3:10, 3:10])

8-element Vector{Float64}:
 0.4710942114589487
 0.4920957209424428
 0.45645878453763855
 0.42768015357353817
 0.42810674673492155
 0.44228276563628016
 0.4418561724748966
 0.42236414648552856

In [188]:
r_square = 1.0 .- diagC*alpha

8-element Vector{Float64}:
 0.057811577082102605
 0.01580855811511439
 0.0870824309247229
 0.14463969285292366
 0.1437865065301569
 0.11543446872743968
 0.11628765505020677
 0.15527170702894288

In [190]:
r = sqrt.(r_square)

8-element Vector{Float64}:
 0.24044038155456043
 0.12573208864531915
 0.295097324496043
 0.3803152545624796
 0.3791919125326342
 0.33975648445237905
 0.34100975799851646
 0.39404531088307965

In [192]:
SEP = sqrt.(diagC.*varE)

8-element Vector{Float64}:
 4.34094096462483
 4.436646124912118
 4.272979216133112
 4.136085848110691
 4.13814812076572
 4.206103972258794
 4.20407503489125
 4.1102999719510915

In [195]:
accuracy = round.([a diagC r_square r SEP], digits=3)

8×5 Matrix{Float64}:
 1.0  0.471  0.058  0.24   4.341
 2.0  0.492  0.016  0.126  4.437
 3.0  0.456  0.087  0.295  4.273
 4.0  0.428  0.145  0.38   4.136
 5.0  0.428  0.144  0.379  4.138
 6.0  0.442  0.115  0.34   4.206
 7.0  0.442  0.116  0.341  4.204
 8.0  0.422  0.155  0.394  4.11